# 03 — Preprocessing Validation

Visualizes EEG signals before and after preprocessing.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.preprocessing.pipeline import EEGPreprocessingPipeline, PreprocessingConfig

# Generate test signal
sfreq = 500.0
t = np.linspace(0, 4, int(4*sfreq))
C = 8
# Simulate multi-channel EEG with alpha + noise
raw_eeg = np.random.randn(C, len(t)) * 30
for i in range(C):
    raw_eeg[i] += 20 * np.sin(2*np.pi*10*t)  # 10Hz alpha
    raw_eeg[i] += 5 * np.sin(2*np.pi*50*t)   # 50Hz powerline

cfg = PreprocessingConfig(target_sfreq=160.0, l_freq=1.0, h_freq=40.0,
                           notch_freq=50.0, apply_car=True, normalization='zscore',
                           segment_len_sec=4.0)
pipeline = EEGPreprocessingPipeline(cfg, orig_sfreq=sfreq)
processed = pipeline(raw_eeg)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(raw_eeg[0, :400], label='Raw', alpha=0.8)
axes[0].set_title('Raw EEG (first channel, first 0.8s @ 500Hz)')
axes[0].set_ylabel('Amplitude (μV)')

axes[1].plot(processed[0, :128], label='Processed', color='orange', alpha=0.8)
axes[1].set_title('After preprocessing (first channel, first 0.8s @ 160Hz)')
axes[1].set_ylabel('Normalized amplitude')

plt.tight_layout()
plt.savefig('../results/figures/preprocessing_comparison.pdf')
plt.show()
print(f'Raw shape: {raw_eeg.shape}, Processed shape: {processed.shape}')